# Shipment Analytics — FreightFox Take-Home Assignment

This notebook explores `shipments.csv`, documents data quality issues, and backs up
every answer in `BUSINESS_ANSWERS.md` with a query or calculation.

**Structure:**
1. Load & clean data
2. Data quality findings (Q4)
3. Q1 — Worst region for on-time delivery
4. Q2 — Freight cost vs. distance, carrier deviation
5. Q3 — Customer-level delivery delays
6. Q5 — Recommended weekly metric


In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so we can import clean_data.py from the project root

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, binomtest
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', None)

from clean_data import load_and_clean

df, quality_report = load_and_clean('../data/shipments.csv')
df.head()

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status,delay_days,flag_impossible_dates,flag_missing_actual_when_completed,is_late_by_date,valid_for_delay_analysis,cost_per_km
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,72921.16,894,Delivered,2.0,False,False,1.0,True,81.567293
1,SHP003362,2026-06-01,2026-06-01,2026-06-04,Kolkata,Raipur,East,FTL,CARR_03,CUST_108,2026-06-04,2026-06-03,2451.90,91,Delivered,-1.0,False,False,0.0,True,26.943956
2,SHP000796,2026-01-17,2026-01-20,2026-01-23,Surat,Jaipur,West,FTL,CARR_05,CUST_108,2026-01-23,NaT,10320.66,395,In-Transit,NaN,False,False,NaN,False,26.128253
3,SHP000291,2026-04-25,2026-04-25,2026-05-01,Nagpur,Bhubaneswar,Central,FTL,CARR_09,CUST_063,2026-05-01,2026-05-02,30242.52,1223,Delayed,1.0,False,False,1.0,True,24.728144
4,SHP003739,2026-01-29,2026-01-30,2026-02-03,Ahmedabad,Hyderabad,West,LTL,CARR_04,CUST_064,2026-02-03,2026-02-01,18434.26,1664,Delivered,-2.0,False,False,0.0,True,11.078281


## 1. Data Quality Findings (backs up Q4)

All cleaning logic lives in `clean_data.py` so the notebook and dashboard stay consistent.

In [2]:
for k, v in quality_report.items():
    print(f"{k}: {v}")

n_raw_rows: 5015
n_exact_duplicates_dropped: 15
n_after_dedup: 5000
delivery_date_equals_promised_pct: 100.0
n_impossible_date_rows: 74
n_completed_missing_actual_date: 682
n_status_vs_date_mismatch: 1742
n_missing_booking_date: 71
n_missing_pickup_date: 87
n_valid_for_delay_analysis: 3444
n_origin_eq_destination: 244


**Key issues found:**
- **15 exact duplicate rows** — dropped.
- **`delivery_date` is 100% redundant** with `promised_delivery_date` (identical in every row) — excluded from analysis; `actual_delivery_date` is the real outcome field.
- **`status` label disagrees with date-derived reality** in a large share of rows (e.g. "Delivered" shipments that were actually late per the dates). All SLA/on-time metrics below are computed from dates, never from `status`.
- **682 "Delivered"/"Delayed" shipments have no `actual_delivery_date` logged** — see next cell, this is concentrated almost entirely in one region.
- **74 rows have logically impossible dates** (delivered before pickup/booking) — excluded from time-based analysis.


In [3]:
# Confirm delivery_date redundancy
print("delivery_date == promised_delivery_date in", quality_report['delivery_date_equals_promised_pct'], "% of rows")

# Confirm status label unreliability
has_delay = df['delay_days'].notna()
mismatch = has_delay & (
    ((df['status']=='Delivered') & (df['delay_days']>0)) |
    ((df['status']=='Delayed') & (df['delay_days']<=0))
)
print(f"Status/date mismatch: {mismatch.sum()} rows out of {has_delay.sum()} with computable delay")

delivery_date == promised_delivery_date in 100.0 % of rows
Status/date mismatch: 1742 rows out of 3518 with computable delay


In [4]:
# Where is the missing actual_delivery_date concentrated?
completed = df[df['status'].isin(['Delivered','Delayed'])]
missing_by_region = completed.groupby('region')['actual_delivery_date'].apply(lambda x: x.isna().sum())
total_by_region = completed.groupby('region').size()
pd.DataFrame({'missing_actual_date': missing_by_region, 'total_completed': total_by_region,
              'pct_missing': (missing_by_region/total_by_region*100).round(1)}).sort_values('pct_missing', ascending=False)

,missing_actual_date,total_completed,pct_missing
region,,,
South,682,807,84.5
Central,0,860,0.0
East,0,835,0.0
North,0,831,0.0
West,0,867,0.0


**Finding:** South region accounts for essentially all of the missing-actual-date rows (84% of its completed shipments have no delivery date logged), vs. 0% in North. This is a data pipeline gap specific to South, not a performance signal — South is excluded from delivery-performance comparisons below.

## 2. Q1 — Which region has the worst on-time delivery performance, and what's driving it?

In [5]:
valid = df[df['valid_for_delay_analysis']]
reliable = valid[valid['region'] != 'South']  # South excluded — unreliable data, see above

region_summary = reliable.groupby('region').agg(
    n=('shipment_id','count'),
    on_time_pct=('delay_days', lambda x: round((x<=0).mean()*100,1)),
    breach_pct=('delay_days', lambda x: round((x>0).mean()*100,1)),
    avg_delay_days=('delay_days','mean'),
).sort_values('breach_pct', ascending=False)
region_summary

,n,on_time_pct,breach_pct,avg_delay_days
region,,,,
Central,836,48.3,51.7,0.552632
North,811,49.6,50.4,0.574599
East,819,50.3,49.7,0.420024
West,854,51.3,48.7,0.272834


In [6]:
# Is the regional spread statistically meaningful?
ct = pd.crosstab(reliable['region'], reliable['delay_days']>0)
chi2, p, dof, exp = chi2_contingency(ct)
print(f"Chi-square test (region vs breach): chi2={chi2:.2f}, p-value={p:.4f}")
print("-> Not statistically significant: we cannot confidently say any region performs worse than another.")

Chi-square test (region vs breach): chi2=1.58, p-value=0.6648
-> Not statistically significant: we cannot confidently say any region performs worse than another.


In [7]:
# Is the real driver carrier, rather than region?
carrier_breach = reliable.groupby('carrier_id').agg(
    n=('shipment_id','count'),
    breach_pct=('delay_days', lambda x: round((x>0).mean()*100,1))
).sort_values('breach_pct', ascending=False)
carrier_breach

,n,breach_pct
carrier_id,,
CARR_02,230,59.1
CARR_07,215,55.8
CARR_13,221,55.7
CARR_08,219,53.9
CARR_06,199,51.8
CARR_03,211,50.7
CARR_04,208,50.5
CARR_01,236,50.0
CARR_05,232,49.6


In [8]:
# Are the worst carriers disproportionately concentrated in Central (the nominal 'worst' region)?
carrier_region_share = pd.crosstab(reliable['carrier_id'], reliable['region'], normalize='index').round(3)*100
carrier_region_share['Central'].sort_values(ascending=False)

carrier_id
CARR_03    30.8
CARR_02    30.0
CARR_11    27.8
CARR_12    26.8
CARR_13    26.2
CARR_01    25.8
CARR_14    25.4
CARR_05    25.0
CARR_09    24.7
CARR_15    24.4
CARR_08    24.2
CARR_06    24.1
CARR_10    24.0
CARR_04    21.2
CARR_07    16.7
Name: Central, dtype: float64

**Answer:** South cannot be evaluated (84% of its completed shipments are missing delivery dates — a data pipeline problem, not a performance one). Among the four regions with reliable data, breach rates range narrowly from 48.7% (West) to 51.7% (Central) — a chi-square test shows this spread is **not statistically significant** (p=0.66). Mode mix and average distance are also near-identical across regions. The real driver is **carrier**, not region: carrier-level breach rates span 44%–59% (a 15-point spread, 5x wider than the regional spread), and this holds consistently regardless of region — bad carriers aren't concentrated in any one place. So the actionable lever is carrier management, not regional operations, and South's broken data pipeline is the most urgent finding here.

## 3. Q2 — Is there a relationship between freight cost and distance? Which carrier(s) deviate?

In [9]:
print("Overall Pearson r (freight_cost vs distance_km):", round(df['freight_cost'].corr(df['distance_km']),3))
print()
for m in df['mode'].unique():
    sub = df[df['mode']==m]
    print(f"{m}: r={sub['freight_cost'].corr(sub['distance_km']):.3f}, n={len(sub)}")

Overall Pearson r (freight_cost vs distance_km): 0.296

PTL: r=0.312, n=985
FTL: r=0.337, n=2033
LTL: r=0.330, n=1982


In [10]:
# Check CARR_07 specifically
carr07 = df[df['carrier_id']=='CARR_07']
others = df[df['carrier_id']!='CARR_07']
print("CARR_07 avg cost/km by mode:")
print(carr07.groupby('mode')['cost_per_km'].mean())
print()
print("All other carriers avg cost/km by mode:")
print(others.groupby('mode')['cost_per_km'].mean())

CARR_07 avg cost/km by mode:
mode
FTL    249.465134
LTL    117.511399
PTL     78.704608
Name: cost_per_km, dtype: float64

All other carriers avg cost/km by mode:
mode
FTL    24.976431
LTL    12.006008
PTL     7.945207
Name: cost_per_km, dtype: float64


In [11]:
# Refit the cost model excluding CARR_07 (the anomaly), then check every carrier's deviation
clean = df[df['carrier_id']!='CARR_07'].copy()
clean['predicted_cost'] = np.nan
for m in clean['mode'].unique():
    mask = clean['mode']==m
    sub = clean[mask]
    slope, intercept = np.polyfit(sub['distance_km'], sub['freight_cost'], 1)
    clean.loc[mask, 'predicted_cost'] = intercept + slope * sub['distance_km']
    print(f"{m}: cost = {intercept:.1f} + {slope:.2f} * distance   (r={sub['freight_cost'].corr(sub['distance_km']):.3f})")

clean['pct_deviation'] = (clean['freight_cost'] - clean['predicted_cost']) / clean['predicted_cost'] * 100
print()
clean.groupby('carrier_id')['pct_deviation'].mean().sort_values(ascending=False).round(2)

FTL: cost = 0.4 + 25.01 * distance   (r=0.985)
LTL: cost = -14.4 + 12.03 * distance   (r=0.985)
PTL: cost = 51.3 + 7.90 * distance   (r=0.984)



carrier_id
CARR_05    1.04
CARR_11    0.34
CARR_01    0.18
CARR_12    0.14
CARR_09    0.07
CARR_14   -0.11
CARR_06   -0.15
CARR_04   -0.19
CARR_02   -0.22
CARR_03   -0.26
CARR_15   -0.30
CARR_08   -0.34
CARR_10   -0.77
CARR_13   -1.11
Name: pct_deviation, dtype: float64

In [12]:
# How consistent is CARR_07's anomaly across ALL of its shipments (not just a few outliers)?
mode_median_cpk = others.groupby('mode')['cost_per_km'].median()
carr07 = carr07.copy()
carr07['ratio_to_median'] = carr07['cost_per_km'] / carr07['mode'].map(mode_median_cpk)
print(carr07['ratio_to_median'].describe())
print("Shipments with ratio > 3x normal median:", (carr07['ratio_to_median']>3).sum(), "out of", len(carr07))

count    342.000000
mean       9.893905
std        1.438573
min        6.949119
25%        8.805021
50%        9.825085
75%       10.864547
max       13.522123
Name: ratio_to_median, dtype: float64
Shipments with ratio > 3x normal median: 342 out of 342


**Answer:** Freight cost is almost perfectly explained by distance once you separate by mode (r≈0.985 for FTL/LTL/PTL) — **except for CARR_07**, whose entire fleet of 342 shipments (100%, not a subset) bills at roughly 7–13x (avg ~10x) the normal rate for its mode, with remarkable consistency (tight ratio distribution). That consistency points to a systematic issue — a cost figure off by ~10x, unit/currency mismatch, or an unlabeled premium tier — worth verifying against the billing source rather than assuming either way. Excluding CARR_07, every other carrier (14 of 15) prices within ~1% of the expected cost curve: there is no other meaningful pricing deviation in the fleet.

## 4. Q3 — Which customer(s) show the most delivery delays? Carrier, region, or something else?

In [13]:
overall_rate = (valid['delay_days']>0).mean()
cust = valid.groupby('customer_id').agg(n=('shipment_id','count'), n_breach=('delay_days', lambda x: (x>0).sum())).reset_index()
cust['breach_pct'] = (cust['n_breach']/cust['n']*100).round(1)
cust['p_value'] = cust.apply(lambda r: binomtest(r['n_breach'], r['n'], overall_rate, alternative='two-sided').pvalue, axis=1)
cust.sort_values('breach_pct', ascending=False).head(10)

,customer_id,n,n_breach,breach_pct,p_value
25,CUST_026,23,17,73.9,0.034690
49,CUST_050,31,22,71.0,0.029450
115,CUST_116,34,24,70.6,0.024307
62,CUST_063,33,23,69.7,0.035083
118,CUST_119,32,22,68.8,0.050104
113,CUST_114,32,22,68.8,0.050104
70,CUST_071,27,18,66.7,0.122080
61,CUST_062,23,15,65.2,0.210041
16,CUST_017,33,21,63.6,0.162758
50,CUST_051,22,14,63.6,0.286281


In [14]:
sig = cust[cust['p_value']<0.05]
print(f"{len(sig)} of {len(cust)} customers are 'significant' at p<0.05")
print("Expected by chance alone at this threshold:", round(len(cust)*0.05,1))
sig.sort_values('breach_pct', ascending=False)

6 of 120 customers are 'significant' at p<0.05
Expected by chance alone at this threshold: 6.0


,customer_id,n,n_breach,breach_pct,p_value
25,CUST_026,23,17,73.9,0.034690
49,CUST_050,31,22,71.0,0.029450
115,CUST_116,34,24,70.6,0.024307
62,CUST_063,33,23,69.7,0.035083
58,CUST_059,36,12,33.3,0.046905
47,CUST_048,39,10,25.6,0.002209


In [15]:
# Do the top-4 raw 'worst' customers share a concentrated carrier or region?
top4 = ['CUST_026','CUST_050','CUST_116','CUST_063']
sub = valid[valid['customer_id'].isin(top4)]
print(pd.crosstab(sub['customer_id'], sub['region']))
print()
print(pd.crosstab(sub['customer_id'], sub['carrier_id']).sum().sort_values(ascending=False).head())

region       Central  East  North  South  West
customer_id                                   
CUST_026           5     5      5      0     8
CUST_050          12     6      3      2     8
CUST_063           5    11      8      1     8
CUST_116           7     4     12      2     9



carrier_id
CARR_02    13
CARR_01    11
CARR_03    11
CARR_08    11
CARR_04    10
dtype: int64


**Answer:** No individual customer is a statistically genuine outlier — with 120 customers tested, ~6 would look "significant" at p<0.05 by chance alone, and that's almost exactly what we observe. The customers with the highest raw breach rates (66–74%) don't share a concentrated carrier or region either. This is most consistent with sampling noise (each customer has only 20–35 shipments here), not a real carrier-, region-, or customer-driven pattern. The robust, actionable levers remain the carrier-level findings from Q1/Q2 (CARR_02's elevated breach rate, CARR_07's cost anomaly) — chasing individual "problem customers" off this snapshot isn't well supported by the data.

## 5. Q5 — One metric to track weekly

**Recommendation: On-time delivery rate, tracked per carrier (not blended network-wide).**

Rationale: this analysis found carrier to be the dominant, statistically real driver of delay
(15pp spread vs. a non-significant 3pp regional spread). A single blended company-wide
on-time % would hide a specific carrier degrading. Tracking it per carrier, weekly, is directly
actionable — reroute volume away from underperforming carriers before it shows up in the
aggregate number.

**Caveat:** this only works if paired with a companion check — % of shipments missing
`actual_delivery_date` — since that's exactly the silent failure that broke South's data and
would otherwise make the on-time metric itself unreliable without anyone noticing.